# Bayesian Optimization for Molecular Design (QM9)

This notebook implements an end-to-end molecular optimization pipeline using:

- QM9 dataset
- SELFIES-based Variational Autoencoder (VAE)
- Gaussian Process surrogate model
- Bayesian Optimization with UCB/LCB acquisition
- PolyChord nested sampling over the VAE latent space

Goal:
- Suggest **5 valid new molecules** that optimize a target molecular property.

> **Environment note:** PolyChordLite contains compiled Fortran/C code and requires a compatible compiler toolchain, including gfortran and gcc (and OpenMPI when MPI parallelization is used). Its official installation instructions are designed around a Unix-style build environment. Therefore, on this Windows machine, pypolychord was installed and run in an Ubuntu Linux Python virtual environment through Windows Subsystem for Linux (WSL). Activate this Ubuntu/WSL virtual environment before running the PolyChord section. See the PolyChordLite installation instructions and Microsoft WSL installation guide.

## STEP 1 - Imports & Configuration

Import the numerical, chemistry, deep-learning, and Gaussian-process libraries used throughout the workflow. The configuration selects a GPU when available, fixes random seeds for reproducibility, uses a 32-dimensional latent space, targets the QM9 `gap` property, and requests five final molecular suggestions.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

from rdkit import Chem
import selfies as sf

from sklearn.preprocessing import StandardScaler

from botorch.models import SingleTaskGP
try:
    from botorch.fit import fit_gpytorch_mll as fit_gpytorch_model
except ImportError:
    from botorch.fit import fit_gpytorch_model
from gpytorch.mlls import ExactMarginalLogLikelihood

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LATENT_DIM = 32
TARGET = "gap"
N_SUGGESTIONS = 5

torch.manual_seed(42)
np.random.seed(42)


## STEP 2 - Load QM9

Load the QM9 CSV file, retain the molecular SMILES strings and selected target property, and remove rows with missing values.

In [4]:
df = pd.read_csv("qm9.csv")[["smiles", TARGET]].dropna().reset_index(drop=True)
df.head()


,smiles,gap
0,C,0.5048
1,N,0.3399
2,O,0.3615
3,C#C,0.3351
4,C#N,0.3796


## STEP 3 - Transform SMILES to SELFIES

Convert each SMILES string into SELFIES, a molecular string representation designed to produce valid molecular structures. Molecules that cannot be encoded are removed.

In [5]:
def smiles_to_selfies(smiles):
    try:
        return sf.encoder(smiles)
    except:
        return None

df["selfies"] = df["smiles"].apply(smiles_to_selfies)
df = df.dropna().reset_index(drop=True)

## STEP 4 - SELFIES Tokenization

Build a vocabulary from all SELFIES tokens, reserve index 0 for padding, create token-to-index lookup tables, and determine the maximum sequence length used by the VAE.

In [6]:
alphabet = sorted({tok for s in df["selfies"] for tok in sf.split_selfies(s)})
alphabet = ["[PAD]"] + alphabet

token_to_idx = {tok: i for i, tok in enumerate(alphabet)}
idx_to_token = {i: tok for tok, i in token_to_idx.items()}

MAX_LEN = max(len(list(sf.split_selfies(s))) for s in df["selfies"])


## STEP 5 - Encode SELFIES to Integer Sequences

Map every SELFIES token to its integer index and pad shorter molecules to `MAX_LEN`. The resulting tensor is the fixed-length input consumed by the neural network.

In [7]:
def selfies_to_tokens(selfies):
    return list(sf.split_selfies(selfies))

def encode_selfies(selfies):
    tokens = selfies_to_tokens(selfies)
    idxs = [token_to_idx[t] for t in tokens]
    idxs += [0] * (MAX_LEN - len(idxs))  # PAD
    return idxs

X_seq = torch.tensor(
    [encode_selfies(s) for s in df["selfies"]],
    dtype=torch.long
)

X_seq.shape


torch.Size([133743, 21])

## STEP 6 - Define SELFIES-VAE Model

Define a variational autoencoder that compresses tokenized molecules into a continuous latent representation and reconstructs SELFIES token probabilities from latent vectors.

**Main parameters:** `vocab_size` is the number of SELFIES tokens; `LATENT_DIM=32` controls the size of the molecular latent vector; the 128-dimensional embedding represents input tokens; the 256-unit GRU encodes and decodes sequences; `fc_mu` and `fc_logvar` describe the latent Gaussian distribution; and `MAX_LEN` fixes the decoded sequence length. The reparameterization step adds sampled Gaussian noise while preserving gradient-based training.

In [8]:
LATENT_DIM = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class SelfiesVAE(nn.Module):
    def __init__(self, vocab_size, latent_dim):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, 128)
        self.encoder = nn.GRU(128, 256, batch_first=True)

        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        self.decoder_input = nn.Linear(latent_dim, 256)
        self.decoder = nn.GRU(128, 256, batch_first=True)
        self.output = nn.Linear(256, vocab_size)

    def encode(self, x):
        emb = self.embed(x)
        _, h = self.encoder(emb)
        h = h.squeeze(0)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.decoder_input(z).unsqueeze(0)
        inputs = torch.zeros((z.size(0), MAX_LEN, 128), device=z.device)
        out, _ = self.decoder(inputs, h)
        return self.output(out)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


## STEP 7 - Train the SELFIES-VAE

Train the VAE to reconstruct each input SELFIES sequence while regularizing its latent space toward a standard normal distribution. The total loss combines token-level cross-entropy reconstruction loss with KL-divergence loss.

**Training parameters:** Adam uses `lr=1e-3`; `BATCH_SIZE=256` controls how many molecules are processed per update; `EPOCHS=30` controls the number of complete passes through the dataset; and padding index 0 is ignored when calculating reconstruction loss.

In [9]:
vae = SelfiesVAE(len(token_to_idx), LATENT_DIM).to(DEVICE)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

X_seq = X_seq.to(DEVICE)

BATCH_SIZE = 256
EPOCHS = 30

for epoch in range(EPOCHS):
    vae.train()
    total_loss = 0

    for i in range(0, len(X_seq), BATCH_SIZE):
        batch = X_seq[i:i+BATCH_SIZE]

        recon, mu, logvar = vae(batch)

        recon_loss = F.cross_entropy(
            recon.view(-1, recon.size(-1)),
            batch.view(-1),
            ignore_index=0
        )

        kl_loss = -0.5 * torch.mean(
            1 + logvar - mu.pow(2) - logvar.exp()
        )

        loss = recon_loss + kl_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.3f}")


Epoch 1/30 | Loss: 976.724
Epoch 2/30 | Loss: 940.790
Epoch 3/30 | Loss: 909.792
Epoch 4/30 | Loss: 883.691
Epoch 5/30 | Loss: 865.572
Epoch 6/30 | Loss: 844.798
Epoch 7/30 | Loss: 820.383
Epoch 8/30 | Loss: 790.267
Epoch 9/30 | Loss: 750.987
Epoch 10/30 | Loss: 716.870
Epoch 11/30 | Loss: 682.726
Epoch 12/30 | Loss: 650.492
Epoch 13/30 | Loss: 626.793
Epoch 14/30 | Loss: 600.135
Epoch 15/30 | Loss: 589.181
Epoch 16/30 | Loss: 570.254
Epoch 17/30 | Loss: 564.302
Epoch 18/30 | Loss: 559.810
Epoch 19/30 | Loss: 548.044
Epoch 20/30 | Loss: 539.809
Epoch 21/30 | Loss: 526.703
Epoch 22/30 | Loss: 518.860
Epoch 23/30 | Loss: 513.460
Epoch 24/30 | Loss: 502.748
Epoch 25/30 | Loss: 497.751
Epoch 26/30 | Loss: 493.972
Epoch 27/30 | Loss: 490.745
Epoch 28/30 | Loss: 489.099
Epoch 29/30 | Loss: 485.472
Epoch 30/30 | Loss: 475.943


### Save the SELFIES_VAE Model

In [10]:
import os

os.makedirs("models", exist_ok=True)

torch.save(
    {
        "model_state_dict": vae.state_dict(),
        "vocab_size": len(token_to_idx),
        "latent_dim": LATENT_DIM,
        "max_len": MAX_LEN,
        "token_to_idx": token_to_idx,
        "idx_to_token": idx_to_token,
    },
    "models/selfies_vae.pt"
)


### Load the SELFIES_VAE Model

In [11]:
checkpoint = torch.load("models/selfies_vae.pt", map_location=DEVICE)

vae = SelfiesVAE(
    vocab_size=checkpoint["vocab_size"],
    latent_dim=checkpoint["latent_dim"]
).to(DEVICE)

vae.load_state_dict(checkpoint["model_state_dict"])
vae.eval()

token_to_idx = checkpoint["token_to_idx"]
idx_to_token = checkpoint["idx_to_token"]
MAX_LEN = checkpoint["max_len"]


## STEP 8 - Encode QM9 into Latent Space

Release training-only GPU memory and pass all QM9 sequences through the trained encoder. The encoder mean `mu` is used as a deterministic 32-dimensional molecular representation. `ENCODE_BATCH_SIZE=256` limits GPU memory usage during inference.

In [12]:
# Keep the full dataset in system RAM and encode it in GPU batches.
# The optimizer is no longer needed and may still hold several GB of GPU state.
import gc

X_seq = X_seq.cpu()
# Remove references to the final training graph and Adam state before inference.
for name in ("optimizer", "recon", "logvar", "recon_loss",
             "kl_loss", "loss", "checkpoint"):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

ENCODE_BATCH_SIZE = 256  # Safe for an 8 GB GPU; use 128 if other apps use VRAM.
vae.eval()
latent_batches = []

with torch.inference_mode():
    for start in range(0, len(X_seq), ENCODE_BATCH_SIZE):
        batch = X_seq[start:start + ENCODE_BATCH_SIZE].to(DEVICE)
        mu, _ = vae.encode(batch)
        latent_batches.append(mu.cpu())

Z = torch.cat(latent_batches, dim=0)
del latent_batches, batch, mu

Z.shape

torch.Size([133743, 32])

## STEP 9 - Prepare Target Property

Standardize the selected molecular property to approximately zero mean and unit variance, then pair those target values with the VAE latent vectors to form the Gaussian-process training data.

In [13]:
y = df[TARGET].values.reshape(-1, 1)

y_scaler = StandardScaler()
y_scaled = y_scaler.fit_transform(y)

train_X = Z.float()
train_Y = torch.tensor(y_scaled, dtype=torch.float32)


## STEP 10 - Train Gaussian Process Surrogate (BoTorch)

Fit an exact single-task Gaussian process that predicts the standardized target property and its uncertainty from a molecular latent vector. This surrogate supplies the posterior mean and variance used by Bayesian optimization.

**Main parameters:** `N_GP=min(3000, len(train_X))` limits exact-GP training cost by randomly selecting at most 3,000 molecules; `SingleTaskGP` models one continuous property; `ExactMarginalLogLikelihood` is the GP training objective; and `fit_gpytorch_model` learns kernel, noise, and other GP hyperparameters by maximizing that objective.

In [14]:
# Exact GP training scales cubically with N and quadratically in memory.
# 3000 points is a practical upper bound for this 8 GB laptop GPU.
N_GP = min(3000, len(train_X))

idx = np.random.choice(len(train_X), size=N_GP, replace=False)

X_gp = train_X[idx].to(DEVICE)
Y_gp = train_Y[idx].to(DEVICE)

gp = SingleTaskGP(X_gp, Y_gp).to(DEVICE)
mll = ExactMarginalLogLikelihood(gp.likelihood, gp)

fit_gpytorch_model(mll)

/tmp/ipykernel_68333/2878813122.py:10: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  gp = SingleTaskGP(X_gp, Y_gp).to(DEVICE)
/mnt/c/Users/ThetK/Desktop/PolyChord/polychord_env/lib/python3.12/site-packages/botorch/models/utils/assorted.py:276: InputDataWarning: Data (input features) is not contained to the unit cube. Please consider min-max scaling the input data.
  check_min_max_scaling(


ExactMarginalLogLikelihood(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (noise_prior): LogNormalPrior()
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (model): SingleTaskGP(
    (likelihood): GaussianLikelihood(
      (noise_covar): HomoskedasticNoise(
        (noise_prior): LogNormalPrior()
        (raw_noise_constraint): GreaterThan(1.000E-04)
      )
    )
    (mean_module): ConstantMean()
    (covar_module): RBFKernel(
      (lengthscale_prior): LogNormalPrior()
      (raw_lengthscale_constraint): GreaterThan(2.500E-02)
    )
    (outcome_transform): Standardize()
  )
)

## STEP 11 - Reference-Aware Bayesian Optimization with PolyChord (UCB/LCB)

Use the GP acquisition function as PolyChord's sampling score to explore promising regions of the VAE latent space. PolyChord maps points from a unit hypercube into the observed latent bounds, samples high-score regions, and returns the best latent vectors as candidate molecules.

The acquisition is explicitly referenced to the best target in the current dataset. For maximization the reference is the highest observed value; for minimization it is the lowest. The GP computes both a confidence bound (UCB or LCB) and the posterior probability that a candidate improves on that reference by at least `MINIMUM_IMPROVEMENT` in the original target units. The PolyChord score combines confidence-bound improvement with `PI_WEIGHT * log(probability of improvement)`, so candidates with a low chance of beating the current best are strongly down-ranked. Increase `PI_WEIGHT` to prioritize reliability over exploration.

**Acquisition parameters:** `OPTIMIZATION_TARGET` chooses minimization or maximization; `ACQUISITION_KIND="auto"` selects LCB for minimization and UCB for maximization; `BETA=2.0` controls confidence-bound exploration; `MINIMUM_IMPROVEMENT=0.0` sets the required improvement over the current best in original target units; and `PI_WEIGHT=1.0` controls how strongly the probability of improvement affects sampling. `N_CANDIDATES=20` is the number of top latent samples retained for decoding.

**PolyChord parameters:** `nDims` is the latent-space size; `nDerived=3` records the confidence bound, probability of improvement, and combined reference-aware score. `nlive`, `num_repeats`, `max_ndead`, and `precision_criterion` control nested sampling. Resume files are disabled and `seed=42` makes sampling reproducible.


In [15]:
from pypolychord import run_polychord
from pypolychord.settings import PolyChordSettings

# User controls:
# - "minimize" uses LCB by default and compares against the lowest observed target.
# - "maximize" uses UCB by default and compares against the highest observed target.
OPTIMIZATION_TARGET = "minimize"  # choose "minimize" or "maximize"
ACQUISITION_KIND = "auto"         # choose "auto", "UCB", or "LCB"
BETA = 2.0
MINIMUM_IMPROVEMENT = 0.0          # required improvement in original target units
PI_WEIGHT = 1.0                    # larger values favor a higher chance of improvement

N_CANDIDATES = 20
POLYCHORD_NLIVE = min(256, max(128, 8 * train_X.shape[-1]))
POLYCHORD_MAX_NDEAD = 2000

optimization_target = OPTIMIZATION_TARGET.lower()
if optimization_target not in {"minimize", "maximize"}:
    raise ValueError("OPTIMIZATION_TARGET must be 'minimize' or 'maximize'")

if ACQUISITION_KIND.lower() == "auto":
    acquisition_kind = "LCB" if optimization_target == "minimize" else "UCB"
else:
    acquisition_kind = ACQUISITION_KIND.upper()

if acquisition_kind not in {"UCB", "LCB"}:
    raise ValueError("ACQUISITION_KIND must be 'auto', 'UCB', or 'LCB'")
expected_acquisition = "UCB" if optimization_target == "maximize" else "LCB"
if acquisition_kind != expected_acquisition:
    raise ValueError(
        f"{optimization_target!r} optimization requires {expected_acquisition}; "
        f"got {acquisition_kind}"
    )
if PI_WEIGHT < 0:
    raise ValueError("PI_WEIGHT must be non-negative")

gp.eval()
gp.likelihood.eval()
GP_DEVICE = next(gp.parameters()).device

nDims = train_X.shape[-1]
nDerived = 3

lower = train_X.min(dim=0).values.detach().cpu().numpy()
upper = train_X.max(dim=0).values.detach().cpu().numpy()
width = np.maximum(upper - lower, 1e-8)

# train_Y and the GP posterior are in StandardScaler units. Use all values in
# the current dataset (not only the GP subsample) to define the best reference.
observed_y = train_Y.squeeze(-1)
reference_scaled = float(
    observed_y.max().item() if optimization_target == "maximize"
    else observed_y.min().item()
)
reference_value = float(y_scaler.inverse_transform([[reference_scaled]])[0, 0])
target_scale = float(y_scaler.scale_[0])
minimum_improvement_scaled = float(MINIMUM_IMPROVEMENT) / target_scale


def prior(cube):
    """Map PolyChord's unit hypercube to the VAE latent-space bounds."""
    cube = np.asarray(cube)
    return lower + cube * width


def acquisition_values(z_np):
    """Return confidence bound, P(improvement), and reference-aware score."""
    z = torch.as_tensor(z_np, dtype=X_gp.dtype, device=GP_DEVICE)
    with torch.inference_mode():
        posterior = gp.posterior(z)
        mean = posterior.mean.squeeze(-1)
        std = posterior.variance.clamp_min(1e-12).sqrt().squeeze(-1)
        beta_scale = BETA ** 0.5

        reference = torch.as_tensor(reference_scaled, dtype=mean.dtype, device=mean.device)
        margin = torch.as_tensor(
            minimum_improvement_scaled, dtype=mean.dtype, device=mean.device
        )
        normal = torch.distributions.Normal(
            torch.zeros((), dtype=mean.dtype, device=mean.device),
            torch.ones((), dtype=mean.dtype, device=mean.device),
        )

        if optimization_target == "maximize":
            confidence_bound = mean + beta_scale * std
            bound_improvement = confidence_bound - reference
            probability_improvement = normal.cdf((mean - reference - margin) / std)
        else:
            confidence_bound = mean - beta_scale * std
            bound_improvement = reference - confidence_bound
            probability_improvement = normal.cdf((reference - margin - mean) / std)

        # log(P(I)) strongly penalizes candidates that are unlikely to beat the
        # current best, while bound_improvement retains UCB/LCB exploration.
        probability_improvement = probability_improvement.clamp(1e-12, 1.0)
        reference_aware_score = (
            bound_improvement + PI_WEIGHT * probability_improvement.log()
        )

    return (
        confidence_bound.detach().cpu().numpy(),
        probability_improvement.detach().cpu().numpy(),
        reference_aware_score.detach().cpu().numpy(),
    )


def loglikelihood(theta):
    confidence_bound, probability_improvement, reference_aware_score = (
        acquisition_values(np.asarray(theta)[None, :])
    )
    bound = float(confidence_bound[0])
    probability = float(probability_improvement[0])
    score = float(reference_aware_score[0])

    # PolyChord always samples high log-likelihood regions. The score is already
    # oriented so that larger means better for both maximization and minimization.
    return score, [bound, probability, score]


base_dir = os.path.join("chains", "polychord_bo")
file_root = f"{TARGET}_{optimization_target}_{acquisition_kind.lower()}_reference_aware"

settings = PolyChordSettings(
    nDims,
    nDerived,
    nlive=POLYCHORD_NLIVE,
    num_repeats=max(64, 5 * nDims),
    max_ndead=POLYCHORD_MAX_NDEAD,
    precision_criterion=1e-3,
    base_dir=base_dir,
    file_root=file_root,
    read_resume=False,
    write_resume=False,
    feedback=1,
    seed=42,
)

output = run_polychord(loglikelihood, nDims, nDerived, settings, prior=prior)

samples_path = os.path.join(base_dir, f"{file_root}_equal_weights.txt")
samples = np.loadtxt(samples_path)
if samples.ndim == 1:
    samples = samples[None, :]

# File format: weight, -2*logL, latent dimensions, then 3 derived values.
logL_samples = -0.5 * samples[:, 1]
theta_samples = samples[:, 2:2 + nDims]
confidence_bound_samples = samples[:, 2 + nDims]
probability_improvement_samples = samples[:, 3 + nDims]
reference_aware_samples = samples[:, 4 + nDims]

best_idx = np.argsort(reference_aware_samples)[::-1][:N_CANDIDATES]
candidates = torch.as_tensor(theta_samples[best_idx], dtype=X_gp.dtype, device=DEVICE)

print(f"Target: {optimization_target}")
print(f"Acquisition: {acquisition_kind}")
print(f"Current dataset reference ({TARGET}): {reference_value:.6g}")
print(f"Required improvement: {MINIMUM_IMPROVEMENT:.6g}")
print(f"PolyChord samples: {len(samples)}")
print("Top candidate diagnostics:")
for rank, sample_idx in enumerate(best_idx[:5], start=1):
    bound_original = float(
        y_scaler.inverse_transform([[confidence_bound_samples[sample_idx]]])[0, 0]
    )
    print(
        f"  {rank}: {acquisition_kind}={bound_original:.6g}, "
        f"P(improvement)={probability_improvement_samples[sample_idx]:.3f}, "
        f"score={reference_aware_samples[sample_idx]:.3f}"
    )

candidates.shape


PolyChord: MPI is already initilised, not initialising, and will not finalize

PolyChord: Next Generation Nested Sampling
copyright: Will Handley, Mike Hobson & Anthony Lasenby
  version: 1.22.3
  release: 22nd Nov 2025
    email: wh260@mrao.cam.ac.uk

Run Settings
nlive    :     256
nDims    :      32
nDerived :       3
Doing Clustering
Synchronous parallelisation
Generating equally weighted posteriors
Generating weighted posteriors
Clustering on posteriors

generating live points



/tmp/ipykernel_68333/2386712877.py:68: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  z = torch.as_tensor(z_np, dtype=X_gp.dtype, device=GP_DEVICE)



all live points generated

Speed  1 =  0.491E-02 seconds
number of repeats:          160
started sampling

__________________
lives      |  256 |
phantoms   |24518 |
posteriors |  257 |
equals     |  209 |
‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
ncluster   =       1 /       1
ndead      =                 257
nposterior =                 257
nequals    =                 211
nlike      =               48884
<nlike>    =         189.95   (           1.19 per slice )
log(Z)     =          -14.38 +/-  0.04
log(Z_1)   =          -14.38 +/-  0.04 (still evaluating)



__________________
lives      |  256 |
phantoms   |34813 |
posteriors |  514 |
equals     |  318 |
‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
ncluster   =       1 /       1
ndead      =                 514
nposterior =                 514
nequals    =                 313
nlike      =              115947
<nlike>    =         261.96   (           1.64 per slice )
log(Z)     =          -13.95 +/-  0.02
log(Z_1)   =          -13.95 +/-  0.02 (still evaluating)



____________

torch.Size([20, 32])

## STEP 12 - Decode Latent → SELFIES → SMILES (Validity Checked)

Decode each candidate latent vector into token probabilities, choose the most likely token at every position, convert the resulting SELFIES string to SMILES, and use RDKit to verify that the decoded molecular structure is valid.

In [16]:
def decode_latent(z):
    z = z.to(next(vae.parameters()).device)
    with torch.inference_mode():
        logits = vae.decode(z)
    tokens = logits.argmax(dim=-1)[0]

    selfies = "".join(
        idx_to_token[i.item()] for i in tokens if i.item() != 0
    )

    try:
        smiles = sf.decoder(selfies)
        mol = Chem.MolFromSmiles(smiles)
        return smiles if mol is not None else None
    except:
        return None


## STEP 13 - Collect 5 Valid Molecular Suggestions

Decode the ranked latent candidates, discard invalid or duplicate molecules, and stop after collecting `N_SUGGESTIONS=5` unique valid SMILES strings.

In [17]:
N_SUGGESTIONS = 5
valid_smiles = []

with torch.no_grad():
    for z in candidates:
        smi = decode_latent(z.unsqueeze(0))
        if smi is not None and smi not in valid_smiles:
            valid_smiles.append(smi)
        if len(valid_smiles) == N_SUGGESTIONS:
            break

valid_smiles


['C#CCN=C1[NH1]C(=O)O1',
 'COOC1CC1C(C)=O',
 'C1CCC=C=CC([NH1])[NH1]1',
 'N1CC1(CCCC)CCC#N',
 'C=C=NOC(=O)CC=O']